# 🛡️ The Contoso Travel Story: Enterprise API Governance

## The Business Challenge

The Contoso Travel MCP server is running in production (Tutorial 16). The **Chief Information Security Officer (CISO)** has concerns:

> *"Our AI agents are calling this MCP server thousands of times per day. How do we:
> 1. **Track** who is calling which tools?
> 2. **Revoke access** if a key is compromised?
> 3. **Audit** all API calls for compliance?"*

The **Solutions Architect** responds:

> *"This is exactly what **Azure API Management (APIM)** is designed for. We put APIM in front of our MCP server as a gateway. It handles all the enterprise concerns while our MCP server stays focused on business logic."*

## 🏗️ Architect's Decision: Why API Gateway for MCP?

```
┌─────────────────────────────────────────────────────────────────────┐
│                    WITHOUT API GATEWAY                              │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ┌──────────┐      Direct Access      ┌──────────────────────────┐│
│   │ Agent A  │ ────────────────────────►│                          ││
│   └──────────┘                         │     MCP Server           ││
│   ┌──────────┐      Direct Access      │     (Container Apps)     ││
│   │ Agent B  │ ────────────────────────►│                          ││
│   └──────────┘                         │  ⚠️ No tracking          ││
│   ┌──────────┐      Direct Access      │  ⚠️ No key revocation   ││
│   │ Agent C  │ ────────────────────────►│  ⚠️ No audit logs        ││
│   └──────────┘                         └──────────────────────────┘│
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                    WITH API GATEWAY (APIM)                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ┌──────────┐                                                      │
│   │ Agent A  │ ─┐                                                   │
│   └──────────┘  │     ┌────────────────────┐     ┌────────────────┐│
│   ┌──────────┐  ├────►│  Azure APIM        │────►│  MCP Server    ││
│   │ Agent B  │ ─┤     │  ✅ Sub keys       │     │  (protected)   ││
│   └──────────┘  │     │  ✅ Audit logs     │     └────────────────┘│
│   ┌──────────┐  │     │  ✅ Monitoring     │                       │
│   │ Agent C  │ ─┘     └────────────────────┘                       │
│   └──────────┘                                                      │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

## 🔐 What APIM Provides (This Tutorial)

| Layer | What It Does | Business Value |
|-------|-------------|----------------|
| **Subscription Keys** | Unique keys per consumer | Revoke access instantly |
| **Request Logging** | Log all API calls | Compliance, debugging |
| **Native MCP Support** | Auto-discovers MCP tools | One-click setup |

> **Note**: Rate limiting, JWT validation, and advanced policies are covered in **Tutorial 19**.

## 📊 APIM Native MCP Support

APIM has a dedicated **MCP Servers** blade that understands the MCP protocol:

```
┌─────────────────────────────────────────────────────────────────────┐
│                    APIM MCP SERVER REGISTRATION                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Traditional API Registration:                                     │
│   • Manual OpenAPI import                                          │
│   • Configure each endpoint separately                              │
│   • No MCP protocol awareness                                       │
│                                                                     │
│   Native MCP Registration (New!):                                   │
│   • Auto-discovers tools from MCP server                           │
│   • Understands JSON-RPC and SSE                                   │
│   • Session-aware routing (per Mcp-Session-Id)                     │
│   • One-click setup                                                 │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

## 🎯 What This Tutorial Covers

| Part | Topic | What You'll Learn |
|------|-------|------------------|
| **1** | Configuration | Set up APIM and MCP backend |
| **2** | Native Registration | Register MCP server in APIM portal |
| **3** | Testing | Verify MCP tools through gateway |
| **4** | Policies Overview | What policies can do (details in Tutorial 19) |
| **5** | AI Foundry | Use with Azure AI agents |

## 👨‍💻 Architecture We're Building

```
┌─────────────────────────────────────────────────────────────────────┐
│                    SECURED MCP ARCHITECTURE                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ┌─────────────────┐                                              │
│   │   AI Agent      │                                              │
│   │   (AI Foundry)  │                                              │
│   └────────┬────────┘                                              │
│            │ MCPTool with headers                                   │
│            │ Ocp-Apim-Subscription-Key: xxx                        │
│            ▼                                                        │
│   ┌─────────────────────────────────────────────────────────────┐  │
│   │              Azure API Management                            │  │
│   │  ┌────────────────────────────────────────────────────────┐ │  │
│   │  │  Default Security:                                      │ │  │
│   │  │  • Validate subscription key                            │ │  │
│   │  │  • Request/response logging                             │ │  │
│   │  │  • (Tutorial 19: JWT, rate limiting)                    │ │  │
│   │  └────────────────────────────────────────────────────────┘ │  │
│   │                          │                                   │  │
│   │                          ▼                                   │  │
│   │  ┌────────────────────────────────────────────────────────┐ │  │
│   │  │  Backend: travel-mcp-server Container App              │ │  │
│   │  └────────────────────────────────────────────────────────┘ │  │
│   └─────────────────────────────────────────────────────────────┘  │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

**Let's add enterprise governance to our MCP server!** 🛡️

# Tutorial 18: Azure API Management for MCP Servers

## Enterprise API Gateway for MCP Architecture

### What You'll Learn

In this tutorial, you'll expose the **FastMCP Travel Server** (deployed in Tutorial 16) through **Azure API Management (APIM)**, adding enterprise-grade security and monitoring.

**Key Concepts:**
- **Native MCP Server registration** - New APIM feature for direct MCP protocol support
- Subscription key authentication
- Centralized logging and monitoring
- AI Foundry MCPTool integration

**Using Existing Resources:**
- **APIM**: `apim-a35tm-aiagents` (already deployed)
- **MCP Server**: `travel-mcp-server` Container App (from Tutorial 16)

**Duration:** 20 minutes

---

### Prerequisites

- Completed Tutorial 16 (FastMCP deployed to Container Apps)
- Azure CLI installed and logged in
- Existing APIM instance (Developer, Basic, Standard, or Premium tier)

### What's Next

After this tutorial, continue to **Tutorial 19** for:
- JWT validation with Entra ID
- Rate limiting policies
- Advanced security configurations

## Part 1: Configuration

Load environment variables and validate the existing resources.

In [7]:
# Configuration - Load from environment variables
import os
from dotenv import load_dotenv

load_dotenv()

# APIM Configuration
APIM_NAME = os.getenv("APIM_NAME")
APIM_RESOURCE_GROUP = os.getenv("APIM_RESOURCE_GROUP")
APIM_GATEWAY_URL = os.getenv("APIM_GATEWAY_URL")

# MCP Server Configuration (from Tutorial 16)
MCP_SERVER_NAME = os.getenv("MCP_SERVER_NAME")
MCP_BACKEND_URL = os.getenv("MCP_BACKEND_URL")

# APIM Subscription Key (set after registering in portal)
APIM_SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY", "")

# MCP Server base path in APIM
# NOTE: Check your actual APIM MCP server registration for the correct path
# Common paths: "mcp", "travel-mcp", or custom path you specified during registration
MCP_BASE_PATH = os.getenv("MCP_BASE_PATH", "mcp")  # Default to "mcp" - your registered path

# Full APIM MCP endpoint
APIM_MCP_ENDPOINT = f"{APIM_GATEWAY_URL}/{MCP_BASE_PATH}/mcp"

# Azure AI Foundry Configuration (for Part 5)
AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "https://gk-agent-framework-project.services.ai.azure.com/api/projects/agentframworkProject")
MODEL_DEPLOYMENT = os.getenv("MODEL_DEPLOYMENT", "gpt-4.1")

print("=" * 70)
print("Tutorial 18: Native MCP Server in Azure API Management")
print("=" * 70)
print(f"""
  APIM Instance:
    Name:           {APIM_NAME}
    Gateway URL:    {APIM_GATEWAY_URL}
    
  MCP Backend (from Tutorial 16):
    Name:           {MCP_SERVER_NAME}
    Backend URL:    {MCP_BACKEND_URL}
    
  APIM MCP Endpoint:
    URL:            {APIM_MCP_ENDPOINT}
    Base Path:      {MCP_BASE_PATH}
    
  Azure AI Foundry:
    Endpoint:       {AZURE_AI_PROJECT_ENDPOINT}
    Model:          {MODEL_DEPLOYMENT}
""")
print("=" * 70)

Tutorial 18: Native MCP Server in Azure API Management

  APIM Instance:
    Name:           apim-a35tm-aiagents
    Gateway URL:    https://apim-a35tm-aiagents.azure-api.net

  MCP Backend (from Tutorial 16):
    Name:           travel-mcp-server
    Backend URL:    https://travel-mcp-server.wittyriver-349f2f87.eastus.azurecontainerapps.io

  APIM MCP Endpoint:
    URL:            https://apim-a35tm-aiagents.azure-api.net/mcp/mcp
    Base Path:      mcp

  Azure AI Foundry:
    Endpoint:       https://gk-agent-framework-project.services.ai.azure.com/api/projects/agentframworkProject
    Model:          gpt-4.1



## Part 2: Register Native MCP Server in APIM

### 🏗️ Architect's Perspective

The architect explains the new APIM MCP feature:

> *"APIM now has native MCP support. Instead of manually configuring endpoints, we register our MCP server once and APIM automatically discovers all the tools. This is a game-changer for managing MCP servers at scale."*

### Why Native MCP Registration?

| Approach | Pros | Cons |
|----------|------|------|
| **Native MCP** (Recommended) | Auto-discovery, MCP-aware policies, session handling | Newer feature |
| **Manual API** | Full control | More configuration, no tool auto-discovery |

### Steps to Register in Azure Portal

1. **Navigate to APIM**: Azure Portal → API Management → your APIM instance

2. **Open MCP Servers blade**: In the left menu under **APIs**, select **MCP Servers**

3. **Create MCP Server**: Click **+ Create MCP server**

4. **Select registration type**: Choose **Expose an existing MCP server**

5. **Configure the MCP server**:
   | Field | Value | Notes |
   |-------|-------|-------|
   | MCP server base URL | `https://travel-mcp-server.<region>.azurecontainerapps.io/mcp` | Your Container App URL |
   | Transport type | Streamable HTTP | Standard for HTTP-based MCP |
   | Name | `travel-mcp-server` | Display name in APIM |
   | Base path | `travel-mcp` | URL path in APIM gateway |
   | Description | Travel booking tools | Help text |

6. **Create**: Click **Create** to register

### What Happens After Registration

```
┌─────────────────────────────────────────────────────────────────────┐
│                    AFTER MCP REGISTRATION                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   APIM automatically:                                               │
│   ✅ Discovers tools from your MCP server (via MCP protocol)       │
│   ✅ Creates the /mcp endpoint                                      │
│   ✅ Enables subscription key authentication                        │
│   ✅ Sets up session-aware routing                                  │
│                                                                     │
│   Your MCP server is now available at:                              │
│   https://<apim-name>.azure-api.net/<base-path>/mcp                │
│                                                                     │
│   Example:                                                          │
│   https://apim-contoso.azure-api.net/travel-mcp/mcp                │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 👨‍💻 Developer Task

After registration, get your subscription key from the APIM portal.

### Get Subscription Key

After registering the MCP server, get the subscription key for authentication:

1. In APIM, go to **Subscriptions** in the left menu
2. Find or create a subscription with access to your MCP server
3. Click **Show/hide keys** to reveal the primary key
4. Add to your `.env` file as `APIM_SUBSCRIPTION_KEY`

In [8]:
# Verify subscription key is configured
if not APIM_SUBSCRIPTION_KEY:
    print("⚠️  APIM_SUBSCRIPTION_KEY not set in .env")
    print("   Please add your subscription key to continue testing.")
    print("")
    print("   To get the key:")
    print(f"   1. Go to Azure Portal → {APIM_NAME} → Subscriptions")
    print("   2. Find your subscription and show the primary key")
    print("   3. Add to .env: APIM_SUBSCRIPTION_KEY=<your-key>")
else:
    print(f"✅ Subscription key configured: {APIM_SUBSCRIPTION_KEY[:8]}...")

# Check if JWT authentication is required (Tutorial 19 adds this)
print("\n" + "=" * 70)
print("⚠️  NOTE: JWT Authentication")
print("=" * 70)
print("""
After completing Tutorial 19, the APIM MCP server will require JWT tokens
in addition to subscription keys. If you see 401 errors mentioning
"Azure AD JWT not present", you have two options:

  1. Temporarily remove JWT validation in APIM to test subscription-key-only
  2. Skip to Tutorial 19c for the complete JWT authentication flow

This tutorial demonstrates subscription-key authentication only.
""")

✅ Subscription key configured: aa9d85fd...

⚠️  NOTE: JWT Authentication

After completing Tutorial 19, the APIM MCP server will require JWT tokens
in addition to subscription keys. If you see 401 errors mentioning
"Azure AD JWT not present", you have two options:

  1. Temporarily remove JWT validation in APIM to test subscription-key-only
  2. Skip to Tutorial 19c for the complete JWT authentication flow

This tutorial demonstrates subscription-key authentication only.



## Part 3: Test MCP Server Through APIM

Now let's test the MCP server through APIM using the Streamable HTTP transport.

### Understanding Streamable HTTP Transport

MCP's Streamable HTTP transport uses:
- **POST /mcp** endpoint for all requests
- **Mcp-Session-Id** header for session management
- **JSON-RPC 2.0** message format

Flow:
1. **Initialize** → Get session ID from response header
2. **tools/list** → List available tools (include session ID)
3. **tools/call** → Call a specific tool (include session ID)

### ⚠️ Authentication Requirements

| Scenario | Headers Required |
|----------|------------------|
| **Subscription key only** (This tutorial) | `Ocp-Apim-Subscription-Key` |
| **With JWT validation** (After Tutorial 19) | `Ocp-Apim-Subscription-Key` + `Authorization: Bearer <JWT>` |

If your APIM has JWT validation enabled (from Tutorial 19), you'll need to acquire a JWT token. 
The cells below will automatically detect and handle both scenarios.

In [9]:
# Test 1: Initialize MCP connection and get session ID
import requests
import json

print("=" * 70)
print("Test 1: Initialize MCP Connection")
print("=" * 70)
print(f"\n  Endpoint: {APIM_MCP_ENDPOINT}")

# Base headers with subscription key
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "Ocp-Apim-Subscription-Key": APIM_SUBSCRIPTION_KEY
}

# Try to acquire JWT token for APIM endpoints that require it (after Tutorial 19)
JWT_TOKEN = None
try:
    from azure.identity import AzureCliCredential
    credential = AzureCliCredential()
    # Get token for Azure Management API (used as audience in APIM JWT policy)
    token = credential.get_token("https://management.azure.com/.default")
    JWT_TOKEN = token.token
    headers["Authorization"] = f"Bearer {JWT_TOKEN}"
    print(f"  ✅ JWT token acquired (for APIM endpoints with JWT validation)")
except Exception as e:
    print(f"  ℹ️  No JWT token (subscription key only): {str(e)[:50]}")

init_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "apim-tutorial", "version": "1.0"}
    }
}

response = requests.post(APIM_MCP_ENDPOINT, json=init_request, headers=headers)

print(f"\n  Status: {response.status_code}")

MCP_SESSION_ID = ""  # Initialize for later cells

if response.status_code == 200:
    # Get session ID from response header
    MCP_SESSION_ID = response.headers.get("Mcp-Session-Id", "")
    print(f"  Session ID: {MCP_SESSION_ID}")
    
    # Parse response - handle SSE format (event: message\ndata: {...})
    response_text = response.text.strip()
    if response_text:
        # Check if SSE format
        if response_text.startswith("event:"):
            # Parse SSE - extract data line
            for line in response_text.split("\n"):
                if line.startswith("data:"):
                    try:
                        result = json.loads(line[5:].strip())
                        if "result" in result:
                            server_info = result["result"].get("serverInfo", {})
                            print(f"\n  ✅ Connected to: {server_info.get('name', 'Unknown')}")
                            print(f"     Version: {server_info.get('version', 'N/A')}")
                            print(f"     Protocol: {result['result'].get('protocolVersion', 'N/A')}")
                            break
                    except json.JSONDecodeError:
                        print(f"  Raw data: {line[5:100]}")
        else:
            try:
                result = json.loads(response_text)
                if "result" in result:
                    server_info = result["result"].get("serverInfo", {})
                    print(f"\n  ✅ Connected to: {server_info.get('name', 'Unknown')}")
                    print(f"     Version: {server_info.get('version', 'N/A')}")
                    print(f"     Protocol: {result['result'].get('protocolVersion', 'N/A')}")
            except json.JSONDecodeError:
                print(f"  Raw response: {response_text[:300]}")
    else:
        print(f"\n  ✅ Session established (empty response body)")
elif response.status_code == 401:
    error_text = response.text
    print(f"  ❌ Authentication Error (401)")
    if "Azure AD JWT" in error_text:
        print(f"\n  ℹ️  This APIM has JWT validation enabled (from Tutorial 19).")
        print(f"     Make sure you're logged into Azure CLI: az login")
        print(f"     Then re-run this cell to acquire a fresh JWT token.")
    else:
        print(f"     Error: {error_text[:200]}")
else:
    print(f"  ❌ Error: {response.text[:200]}")

print("\n" + "=" * 70)

Test 1: Initialize MCP Connection

  Endpoint: https://apim-a35tm-aiagents.azure-api.net/mcp/mcp
  ✅ JWT token acquired (for APIM endpoints with JWT validation)

  Status: 200
  Session ID: 

  ✅ Connected to: Travel Booking Server
     Version: 2.13.3
     Protocol: 2024-11-05



In [11]:
# Test 2: List available MCP tools
print("=" * 70)
print("Test 2: List Available Tools")
print("=" * 70)

# Build headers - include session ID if available
headers_with_session = {**headers}
if MCP_SESSION_ID:
    headers_with_session["Mcp-Session-Id"] = MCP_SESSION_ID

tools_request = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/list"
}

response = requests.post(APIM_MCP_ENDPOINT, json=tools_request, headers=headers_with_session)

if response.status_code == 200:
    response_text = response.text.strip()
    # Parse SSE format
    for line in response_text.split("\n"):
        if line.startswith("data:"):
            try:
                result = json.loads(line[5:].strip())
                if "result" in result:
                    tools = result["result"].get("tools", [])
                    print(f"\n  ✅ Found {len(tools)} tools:")
                    print("")
                    for tool in tools:
                        print(f"    📦 {tool['name']}")
                        desc = tool.get('description', 'No description')
                        print(f"       {desc[:80]}...")
                        print("")
                    break
            except json.JSONDecodeError:
                print(f"  Parse error: {line[5:100]}")
else:
    print(f"  ❌ Error: {response.status_code} - {response.text[:200]}")

print("=" * 70)

Test 2: List Available Tools

  ✅ Found 4 tools:

    📦 search_flights
       Search for available flights between two cities.

Args:
    origin: Departure ci...

    📦 check_hotel_availability
       Check hotel room availability in a specific location.

Args:
    location: City ...

    📦 convert_currency
       Convert amount from one currency to another.

Args:
    amount: Amount to conver...

    📦 get_server_info
       Get information about this MCP server...



In [12]:
# Test 3: Call a tool - Search for flights
print("=" * 70)
print("Test 3: Call Tool - search_flights")
print("=" * 70)

# Build headers - include session ID if available
headers_with_session = {**headers}
if MCP_SESSION_ID:
    headers_with_session["Mcp-Session-Id"] = MCP_SESSION_ID

call_request = {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "search_flights",
        "arguments": {
            "origin": "SEA",
            "destination": "NRT",
            "departure_date": "2025-01-15"
        }
    }
}

print(f"\n  Searching flights: SEA → NRT on 2025-01-15")

response = requests.post(APIM_MCP_ENDPOINT, json=call_request, headers=headers_with_session)

if response.status_code == 200:
    response_text = response.text.strip()
    # Parse SSE format
    for line in response_text.split("\n"):
        if line.startswith("data:"):
            try:
                result = json.loads(line[5:].strip())
                if "result" in result:
                    content = result["result"].get("content", [])
                    if content:
                        print(f"\n  ✅ Tool response:")
                        for item in content:
                            if item.get("type") == "text":
                                # Parse and display flight results
                                text = item.get("text", "")
                                try:
                                    flights = json.loads(text)
                                    print(f"\n  Found {len(flights)} flights:")
                                    for flight in flights[:3]:
                                        print(f"    ✈️  {flight.get('airline', 'Unknown')} {flight.get('flight_number', '')}")
                                        print(f"       Departs: {flight.get('departure_time', '')} → Arrives: {flight.get('arrival_time', '')}")
                                        print(f"       Price: ${flight.get('price', 'N/A')}")
                                        print("")
                                except json.JSONDecodeError:
                                    print(f"    {text[:200]}")
                elif "error" in result:
                    print(f"  ❌ Tool error: {result['error']}")
                break
            except json.JSONDecodeError:
                print(f"  Parse error: {line[5:100]}")
else:
    print(f"  ❌ Error: {response.status_code} - {response.text[:200]}")

print("\n" + "=" * 70)

Test 3: Call Tool - search_flights

  Searching flights: SEA → NRT on 2025-01-15

  ✅ Tool response:

  Found 3 flights:
    ✈️  Delta UA998
       Departs: 2025-01-15 14:30 → Arrives: 2025-01-15 22:30
       Price: $768.4

    ✈️  American DL103
       Departs: 2025-01-15 15:45 → Arrives: 2025-01-15 22:45
       Price: $1080.22

    ✈️  British Airways DL685
       Departs: 2025-01-15 07:45 → Arrives: 2025-01-15 19:45
       Price: $1014.42




In [13]:
# Test 4: Call another tool - Convert currency
print("=" * 70)
print("Test 4: Call Tool - convert_currency")
print("=" * 70)

# Build headers - include session ID if available
headers_with_session = {**headers}
if MCP_SESSION_ID:
    headers_with_session["Mcp-Session-Id"] = MCP_SESSION_ID

currency_request = {
    "jsonrpc": "2.0",
    "id": 4,
    "method": "tools/call",
    "params": {
        "name": "convert_currency",
        "arguments": {
            "amount": 1000,
            "from_currency": "USD",
            "to_currency": "JPY"
        }
    }
}

print(f"\n  Converting: $1,000 USD → JPY")

response = requests.post(APIM_MCP_ENDPOINT, json=currency_request, headers=headers_with_session)

if response.status_code == 200:
    response_text = response.text.strip()
    # Parse SSE format
    for line in response_text.split("\n"):
        if line.startswith("data:"):
            try:
                result = json.loads(line[5:].strip())
                if "result" in result:
                    content = result["result"].get("content", [])
                    if content:
                        for item in content:
                            if item.get("type") == "text":
                                text = item.get("text", "")
                                try:
                                    conversion = json.loads(text)
                                    print(f"\n  ✅ Conversion result:")
                                    print(f"     {conversion.get('original_amount', '')} {conversion.get('from_currency', '')}")
                                    print(f"     = {conversion.get('converted_amount', '')} {conversion.get('to_currency', '')}")
                                    print(f"     Rate: {conversion.get('exchange_rate', '')}")
                                except json.JSONDecodeError:
                                    print(f"    {text}")
                break
            except json.JSONDecodeError:
                print(f"  Parse error: {line[5:100]}")
else:
    print(f"  ❌ Error: {response.status_code}")

print("\n" + "=" * 70)

Test 4: Call Tool - convert_currency

  Converting: $1,000 USD → JPY

  ✅ Conversion result:
     1000.0 
     = 110000.0 
     Rate: 110.0



## Part 4: APIM Policies Overview

### 🏗️ Architect's Perspective

> *"APIM policies let us add security, rate limiting, and transformation without changing our MCP server code. The detailed policy configuration is covered in Tutorial 19."*

### What APIM Policies Can Do

| Policy Type | Purpose | Covered In |
|-------------|---------|------------|
| **Subscription Keys** | API key authentication | ✅ This tutorial (auto-enabled) |
| **Rate Limiting** | Prevent abuse | Tutorial 19 |
| **JWT Validation** | Entra ID authentication | Tutorial 19 |
| **IP Filtering** | Network-level security | Tutorial 19 |

### Default Security (Native MCP Registration)

When you register an MCP server natively in APIM, it automatically enables:
- ✅ Subscription key requirement
- ✅ Session-aware routing (Mcp-Session-Id)
- ✅ Request/response logging

For advanced policies (JWT, rate limiting, IP filtering), see **Tutorial 19**.

## Part 5: Using with AI Foundry Agent Service

Use the APIM-protected MCP server with Azure AI Foundry agents using the `MCPTool` SDK.

In [16]:
# Using APIM MCP endpoint with Azure AI Foundry Agent
# Pattern aligned with Tutorial 19c

import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool

load_dotenv()

# Configuration
print("=" * 70)
print("AI Foundry Agent with APIM-Protected MCP Server")
print("=" * 70)
print(f"""
  APIM MCP Endpoint: {APIM_MCP_ENDPOINT}
  AI Project: {AZURE_AI_PROJECT_ENDPOINT}
  Model: {MODEL_DEPLOYMENT}
""")

# Create project client
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=credential
)

# Build MCP headers - include JWT if required (after Tutorial 19)
mcp_headers = [f"Ocp-Apim-Subscription-Key:{APIM_SUBSCRIPTION_KEY}"]
try:
    # Get JWT token for APIM endpoints with JWT validation enabled
    token = credential.get_token("https://management.azure.com/.default")
    mcp_headers.append(f"Authorization:Bearer {token.token}")
    print("  ✅ JWT token acquired for APIM authentication")
except Exception as e:
    print(f"  ℹ️  Using subscription key only: {str(e)[:50]}")

# Create MCPTool pointing to APIM endpoint
mcp_tool = MCPTool(
    server_url=APIM_MCP_ENDPOINT,
    headers=mcp_headers
)

print(f"\n  Server URL: {mcp_tool.server_url}")
print(f"  Headers configured: {len(mcp_headers)}")

print("\n" + "=" * 70)
print("MCPTool Ready for Agent Integration")
print("=" * 70)
print("""
See Tutorial 19c for a complete working example with:
  • Agent creation with MCPTool
  • Thread-based conversations
  • Full MCP tool execution
""")

AI Foundry Agent with APIM-Protected MCP Server

  APIM MCP Endpoint: https://apim-a35tm-aiagents.azure-api.net/mcp/mcp
  AI Project: https://gk-agent-framework-project.services.ai.azure.com/api/projects/agentframworkProject
  Model: gpt-4.1

  ✅ JWT token acquired for APIM authentication

  Server URL: https://apim-a35tm-aiagents.azure-api.net/mcp/mcp
  Headers configured: 2

MCPTool Ready for Agent Integration

See Tutorial 19c for a complete working example with:
  • Agent creation with MCPTool
  • Thread-based conversations
  • Full MCP tool execution



## Summary: What Contoso Travel Achieved

### 🎯 The Story So Far

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CONTOSO TRAVEL - MILESTONE 4                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ✅ Tutorial 15: Built FastMCP server with travel tools           │
│   ✅ Tutorial 16: Deployed to Azure Container Apps                 │
│   ✅ Tutorial 17: Added Logic Apps for SaaS connectors             │
│   ✅ Tutorial 18 (This): Added APIM Gateway                        │
│      • Native MCP server registration                              │
│      • Subscription key authentication                              │
│      • Centralized monitoring and logging                          │
│                                                                     │
│   ⏳ Next: Complete security with JWT + policies (Tutorial 19)     │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 📋 What This Tutorial Covered

| Topic | Status |
|-------|--------|
| Native MCP server registration in APIM | ✅ Done |
| Subscription key authentication | ✅ Done |
| Testing MCP through APIM gateway | ✅ Done |
| AI Foundry MCPTool configuration | ✅ Done |

### 🔜 What's Next

| Tutorial | What You'll Learn |
|----------|------------------|
| **19** | JWT validation, rate limiting, advanced policies |
| **19c** | Complete security flow with working agent example |

### Key Documentation

- [About MCP servers in Azure API Management](https://learn.microsoft.com/azure/api-management/mcp-server-overview)
- [Expose an existing MCP server](https://learn.microsoft.com/azure/api-management/expose-existing-mcp-server)

---

**Congratulations!** Your MCP server is now behind an API gateway. Continue to Tutorial 19 for complete security with JWT authentication. 🛡️